## Import Required Libraries

In [1]:
import os
import re
import joblib
import warnings
import numpy as np
import pandas as pd

from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from imblearn.over_sampling import RandomOverSampler

import nltk
from nltk.corpus import stopwords

warnings.filterwarnings("ignore")

print("Libraries imported successfully!")

Libraries imported successfully!


## Observation
All required libraries for preprocessing, text cleaning, feature engineering, and class balancing were imported successfully.

## Download Stopwords

In [2]:
nltk.download("stopwords")
stop_words = set(stopwords.words("english"))

print("Stopwords loaded successfully!")

Stopwords loaded successfully!


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## Observation
The English stopwords corpus was downloaded successfully and is now available for text cleaning.

## Load Dataset

In [3]:
train_df = pd.read_csv("../data/train_data.csv")

print("Dataset loaded successfully!")
print("Shape of dataset:", train_df.shape)

Dataset loaded successfully!
Shape of dataset: (4000, 8)


## Observation
The Amazon reviews dataset was loaded successfully. It contains customer review information and sentiment labels that will be used for sentiment classification.

## Display Dataset Preview

In [4]:
train_df.head()

,Name of the product,Product Brand,categories,primaryCategories,reviews.date,reviews.text,reviews.title,sentiment
0,"All-New Fire HD 8 Tablet, 8"" HD Display, Wi-Fi...",Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...",Electronics,2016-12-26T00:00:00.000Z,Purchased on Black FridayPros - Great Price (e...,Powerful tablet,Positive
1,Amazon - Echo Plus w/ Built-In Hub - Silver,Amazon,"Amazon Echo,Smart Home,Networking,Home & Tools...","Electronics,Hardware",2018-01-17T00:00:00.000Z,I purchased two Amazon in Echo Plus and two do...,Amazon Echo Plus AWESOME,Positive
2,Amazon Echo Show Alexa-enabled Bluetooth Speak...,Amazon,"Amazon Echo,Virtual Assistant Speakers,Electro...","Electronics,Hardware",2017-12-20T00:00:00.000Z,Just an average Alexa option. Does show a few ...,Average,Neutral
3,"Fire HD 10 Tablet, 10.1 HD Display, Wi-Fi, 16 ...",Amazon,"eBook Readers,Fire Tablets,Electronics Feature...","Office Supplies,Electronics",2017-08-04T00:00:00.000Z,"very good product. Exactly what I wanted, and ...",Greattttttt,Positive
4,"Brand New Amazon Kindle Fire 16gb 7"" Ips Displ...",Amazon,"Computers/Tablets & Networking,Tablets & eBook...",Electronics,2017-01-23T00:00:00.000Z,This is the 3rd one I've purchased. I've bough...,Very durable!,Positive


## Observation
The dataset preview helps verify the available columns and gives an initial understanding of the review and sentiment data.

## Check Dataset Information

In [5]:
train_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4000 entries, 0 to 3999
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   Name of the product  4000 non-null   str  
 1   Product Brand        4000 non-null   str  
 2   categories           4000 non-null   str  
 3   primaryCategories    4000 non-null   str  
 4   reviews.date         4000 non-null   str  
 5   reviews.text         4000 non-null   str  
 6   reviews.title        3990 non-null   str  
 7   sentiment            4000 non-null   str  
dtypes: str(8)
memory usage: 2.2 MB


## Observation
The dataset information provides an overview of column data types and missing values, which helps identify the preprocessing steps required before model training.

## Check Missing Values

In [6]:
train_df.isnull().sum()

Name of the product     0
Product Brand           0
categories              0
primaryCategories       0
reviews.date            0
reviews.text            0
reviews.title          10
sentiment               0
dtype: int64

## Observation
The missing value summary shows which columns contain null values and helps determine the appropriate strategy for handling incomplete data.

## Handle Missing Values

In [7]:
train_df["reviews.title"] = train_df["reviews.title"].fillna("No Title")
train_df["reviews.text"] = train_df["reviews.text"].fillna("")

print("Missing values handled successfully!")
print(train_df[["reviews.title", "reviews.text"]].isnull().sum())

Missing values handled successfully!
reviews.title    0
reviews.text     0
dtype: int64


## Observation
Missing values in the review title and review text columns were handled successfully. This ensures that the text cleaning and vectorization steps can run without interruption.

## Remove Duplicate Records

In [8]:
print("Duplicate rows before removal:", train_df.duplicated().sum())

train_df.drop_duplicates(inplace=True)

print("Duplicate rows after removal:", train_df.duplicated().sum())
print("Shape after duplicate removal:", train_df.shape)

Duplicate rows before removal: 58
Duplicate rows after removal: 0
Shape after duplicate removal: (3942, 8)


## Observation
Duplicate records were removed successfully. This helps improve data quality and prevents the model from learning repeated patterns from duplicate reviews.

## Define Text Cleaning Function

In [22]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)   # remove URLs
    text = re.sub(r"[^a-zA-Z\s]", " ", text)              # keep only letters
    text = re.sub(r"\s+", " ", text).strip()              # remove extra spaces

    words = [
        word for word in text.split()
        if word not in stop_words and len(word) > 1
    ]

    return " ".join(words)

## Observation
A text cleaning function was created successfully. It removes noise from the review text and keeps only meaningful words for sentiment analysis.

## Create Cleaned Review Column

In [10]:
train_df["cleaned_review"] = train_df["reviews.text"].apply(clean_text)

train_df[["reviews.text", "cleaned_review"]].head()

,reviews.text,cleaned_review
0,Purchased on Black FridayPros - Great Price (e...,purchased black fridaypros great price even sa...
1,I purchased two Amazon in Echo Plus and two do...,purchased two amazon echo plus two dots plus f...
2,Just an average Alexa option. Does show a few ...,average alexa option show things screen still ...
3,"very good product. Exactly what I wanted, and ...",good product exactly wanted good price
4,This is the 3rd one I've purchased. I've bough...,rd one purchased bought one nieces case compar...


## Observation
The `cleaned_review` column was created successfully. It contains the preprocessed version of customer reviews, which will be used for feature extraction and model training.

## Remove Empty Cleaned Reviews

In [11]:
print("Rows before removing empty cleaned reviews:", train_df.shape[0])

train_df = train_df[train_df["cleaned_review"].str.strip() != ""]

print("Rows after removing empty cleaned reviews:", train_df.shape[0])

Rows before removing empty cleaned reviews: 3942
Rows after removing empty cleaned reviews: 3942


## Observation
Rows with empty cleaned reviews were removed successfully. This ensures that only meaningful review text is used for vectorization and model training.

## Create Review Length Feature

In [12]:
train_df["review_length"] = train_df["cleaned_review"].apply(lambda x: len(x.split()))

train_df["review_length"].describe()

count    3942.000000
mean       15.093861
std        19.287823
min         1.000000
25%         7.000000
50%        10.000000
75%        17.000000
max       708.000000
Name: review_length, dtype: float64

## Observation
The `review_length` feature was created successfully. It represents the number of words in each cleaned review and may help capture review-level sentiment patterns.

## Encode Sentiment Labels

In [13]:
label_encoder = LabelEncoder()

train_df["sentiment_encoded"] = label_encoder.fit_transform(train_df["sentiment"])

print("Sentiment classes:", label_encoder.classes_)
train_df[["sentiment", "sentiment_encoded"]].head()

Sentiment classes: ['Negative' 'Neutral' 'Positive']


,sentiment,sentiment_encoded
0,Positive,2
1,Positive,2
2,Neutral,1
3,Positive,2
4,Positive,2


## Observation
The sentiment labels were encoded successfully into numerical format. This encoded target variable will be used during model training.

## Check Sentiment Distribution

In [14]:
train_df["sentiment"].value_counts()

sentiment
Positive    3694
Neutral      158
Negative      90
Name: count, dtype: int64

## Observation
The sentiment distribution shows the number of Positive, Neutral, and Negative reviews in the dataset. If one class dominates, balancing is required to improve model fairness and performance.

## Apply TF-IDF Vectorization

In [15]:
tfidf = TfidfVectorizer(max_features=5000)

X = tfidf.fit_transform(train_df["cleaned_review"])
y = train_df["sentiment_encoded"]

print("TF-IDF feature shape:", X.shape)
print("Target shape:", y.shape)

TF-IDF feature shape: (3942, 4619)
Target shape: (3942,)


## Observation
The cleaned review text was successfully transformed into TF-IDF numerical features. These features will be used as input for sentiment classification models.

## Balance the Dataset

In [16]:
ros = RandomOverSampler(random_state=42)

X_balanced, y_balanced = ros.fit_resample(X, y)

print("Original X shape:", X.shape)
print("Balanced X shape:", X_balanced.shape)
print("Original y shape:", y.shape)
print("Balanced y shape:", y_balanced.shape)

Original X shape: (3942, 4619)
Balanced X shape: (11082, 4619)
Original y shape: (3942,)
Balanced y shape: (11082,)


## Observation
The dataset was balanced successfully using RandomOverSampler. This oversampling technique increased the minority class samples so that all sentiment classes now contain an equal number of reviews for model training.

## Check Balanced Class Distribution

In [17]:
balanced_counts = pd.Series(y_balanced).value_counts().sort_index()
balanced_counts.index = label_encoder.inverse_transform(balanced_counts.index)

balanced_counts

Negative    3694
Neutral     3694
Positive    3694
Name: count, dtype: int64

## Observation
The balanced class distribution confirms that all sentiment classes now contain equal numbers of reviews, making the dataset more suitable for model training.

## Save Processed Dataset

In [18]:
os.makedirs("../data", exist_ok=True)

train_df.to_csv("../data/processed_data.csv", index=False)

print("Processed dataset saved successfully!")

Processed dataset saved successfully!


## Observation
The processed dataset was saved successfully. It includes the cleaned review text, review length feature, and encoded sentiment labels.

## Save TF-IDF Vectorizer and Label Encoder

In [19]:
os.makedirs("../models", exist_ok=True)

joblib.dump(tfidf, "../models/tfidf_vectorizer.pkl")
joblib.dump(label_encoder, "../models/label_encoder.pkl")

print("TF-IDF vectorizer saved successfully!")
print("Label encoder saved successfully!")

TF-IDF vectorizer saved successfully!
Label encoder saved successfully!


## Observation
The TF-IDF vectorizer and Label Encoder were saved successfully. These files are required for transforming review text and decoding predictions in later stages of the project.

## Save Balanced Feature Matrix and Target Labels

In [20]:
joblib.dump(X_balanced, "../data/X_balanced.pkl")
joblib.dump(y_balanced, "../data/y_balanced.pkl")

print("Balanced feature matrix saved successfully!")
print("Balanced target labels saved successfully!")

Balanced feature matrix saved successfully!
Balanced target labels saved successfully!


## Observation
The balanced TF-IDF feature matrix and balanced target labels were saved successfully. These files will be used directly in Notebook 3 for model training.

## Verify Saved Files

In [21]:
loaded_tfidf = joblib.load("../models/tfidf_vectorizer.pkl")
loaded_label_encoder = joblib.load("../models/label_encoder.pkl")
loaded_X_balanced = joblib.load("../data/X_balanced.pkl")
loaded_y_balanced = joblib.load("../data/y_balanced.pkl")

print("TF-IDF vocabulary size:", len(loaded_tfidf.vocabulary_))
print("Label classes:", loaded_label_encoder.classes_)
print("Loaded X_balanced shape:", loaded_X_balanced.shape)
print("Loaded y_balanced shape:", loaded_y_balanced.shape)

TF-IDF vocabulary size: 4619
Label classes: ['Negative' 'Neutral' 'Positive']
Loaded X_balanced shape: (11082, 4619)
Loaded y_balanced shape: (11082,)


## Observation
All preprocessing files were loaded successfully, confirming that the saved TF-IDF vectorizer, Label Encoder, and balanced training files are ready for model training and deployment.

# Conclusion

In this notebook, the Amazon review dataset was cleaned and prepared for machine learning. Missing values and duplicates were handled, review text was cleaned, review length was engineered as a feature, sentiment labels were encoded, TF-IDF vectorization was applied, and the class imbalance problem was addressed using oversampling. Finally, all required preprocessing files were saved for use in Notebook 3 and the Streamlit sentiment analysis application.